In [1]:
import os
import numpy as np
import random
import warnings
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from skopt import BayesSearchCV
from skopt.space import Integer, Real

# 固定随机种子
os.environ['PYTHONHASHSEED'] = str(1)
np.random.seed(1)
random.seed(1)
warnings.filterwarnings("ignore")

# 1. 数据加载
data = pd.read_excel('dataset.xlsx')
data.rename(columns={"C0": r"C$_0$"}, inplace=True)

X = data.iloc[:, :-1]
y = data.iloc[:, -1]

# 确保数据为数值型
if not np.issubdtype(X.dtypes, np.number):
    X = X.apply(pd.to_numeric, errors='coerce')
if not np.issubdtype(y.dtype, np.number):
    y = pd.to_numeric(y, errors='coerce')

# 检查 NaN/Inf
if X.isnull().any().any() or y.isnull().any():
    raise ValueError("Dataset contains invalid values (NaN or inf). Please check your data.")

# 2. 数据预处理
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.3, random_state=1
)

# 3. 参数优化 (Bayesian Optimization)
sgb_model = GradientBoostingRegressor(random_state=1)

# 注意：subsample < 1.0 才是 "Stochastic Gradient Boosting"
param_spaces = {
    'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'max_depth': Integer(3, 10),
    'n_estimators': Integer(100, 500),
    'min_samples_split': Integer(2, 20),
    'min_samples_leaf': Integer(1, 20),
    'subsample': Real(0.5, 0.9, prior='uniform'),  # 随机采样比例
    'max_features': Real(0.5, 1.0, prior='uniform') # 特征随机采样比例
}

optimizer = BayesSearchCV(
    estimator=sgb_model,
    search_spaces=param_spaces,
    n_iter=32,
    scoring='neg_mean_squared_error',
    cv=5,
    random_state=1
)

optimizer.fit(X_train, y_train)

# 4. 用最优参数训练
best_params = optimizer.best_params_
print(f'Best parameters: {best_params}')
sgb_optimized = GradientBoostingRegressor(**best_params, random_state=1)
sgb_optimized.fit(X_train, y_train)

# 5. 模型评估
def evaluate_model(model, X_train, X_test, y_train, y_test):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    metrics = {
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'Train R^2': r2_score(y_train, y_pred_train),
        'Test R^2': r2_score(y_test, y_pred_test),
        'Train MAE': mean_absolute_error(y_train, y_pred_train),
        'Test MAE': mean_absolute_error(y_test, y_pred_test)
    }
    return metrics

results = evaluate_model(sgb_optimized, X_train, X_test, y_train, y_test)
for metric, value in results.items():
    print(f'{metric}: {value}')

# 6. 导出结果
def export_results(y_train, y_pred_train, y_test, y_pred_test, filename='Results_StochasticGB.xlsx'):
    with pd.ExcelWriter(filename) as writer:
        train_results_df = pd.DataFrame({'y_true': y_train, 'y_pred': y_pred_train})
        test_results_df = pd.DataFrame({'y_true': y_test, 'y_pred': y_pred_test})

        train_results_df.to_excel(writer, sheet_name='Train Results', index=False)
        test_results_df.to_excel(writer, sheet_name='Test Results', index=False)

export_results(
    y_train,
    sgb_optimized.predict(X_train),
    y_test,
    sgb_optimized.predict(X_test)
)


Best parameters: OrderedDict([('learning_rate', 0.015290982092361401), ('max_depth', 3), ('max_features', 1.0), ('min_samples_leaf', 1), ('min_samples_split', 4), ('n_estimators', 500), ('subsample', 0.6212840538276965)])
Train RMSE: 2.0758251955363156
Test RMSE: 3.9198249538038175
Train R^2: 0.9972032236723868
Test R^2: 0.9910250385749508
Train MAE: 1.6448286760580844
Test MAE: 2.8893239795309142
